# CAPSTONE PROJECT: House Price Prediction

## Regression Problem

---

## Project Overview

**Objective:** Build a machine learning model to predict house prices based on various features.

**Business Context:** Real estate companies need accurate price predictions to:
- Help buyers make informed decisions
- Assist sellers in pricing their properties
- Support investment decisions

**Dataset:** Kaggle House Prices Dataset (or similar housing datasets)

---

## Project Structure

```
1. Problem Definition
2. Data Collection
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Feature Engineering
6. Model Building
7. Model Evaluation
8. Hyperparameter Tuning
9. Final Model & Predictions
10. Conclusions & Recommendations
```

---

## Documentation: How to Approach This Capstone

### Step-by-Step Guide

#### 1. Problem Definition
- Clearly state what you're predicting
- Define success metrics (RMSE, R², MAE)
- Understand the business impact

#### 2. Data Collection
- **Kaggle**: Search for "House Prices" or "Real Estate" datasets
- **HuggingFace**: `load_dataset("house_prices")` or similar
- **Sklearn**: `fetch_california_housing()`

#### 3. EDA Checklist
- [ ] Check data shape and types
- [ ] Identify missing values
- [ ] Analyze target variable distribution
- [ ] Examine feature distributions
- [ ] Calculate correlations
- [ ] Identify outliers
- [ ] Visualize relationships

#### 4. Preprocessing Checklist
- [ ] Handle missing values
- [ ] Encode categorical variables
- [ ] Scale numerical features
- [ ] Handle outliers
- [ ] Split data (train/validation/test)

#### 5. Modeling Checklist
- [ ] Start with baseline model
- [ ] Try multiple algorithms
- [ ] Use cross-validation
- [ ] Compare performance

---

## Part 1: Setup and Data Collection

In [ ]:
# Install required packages
# !pip install numpy pandas matplotlib seaborn scikit-learn xgboost

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully!")

In [ ]:
# Option 1: Load from Kaggle
# First, setup Kaggle API:
# !pip install kaggle
# Upload your kaggle.json file
# !kaggle competitions download -c house-prices-advanced-regression-techniques

# Option 2: Load from scikit-learn (California Housing)
from sklearn.datasets import fetch_california_housing

# Load data
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target  # Price in $100,000s

print(f"Dataset Shape: {df.shape}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"\nTarget: Price (Median house value in $100,000s)")
df.head()

In [ ]:
# Alternative: Create a more realistic dataset with categorical features
# This demonstrates handling mixed data types

np.random.seed(42)
n_samples = len(df)

# Add some categorical features
df['PropertyType'] = np.random.choice(['House', 'Condo', 'Townhouse'], n_samples, p=[0.5, 0.3, 0.2])
df['Neighborhood'] = np.random.choice(['Urban', 'Suburban', 'Rural'], n_samples, p=[0.4, 0.45, 0.15])
df['HasPool'] = np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
df['HasGarage'] = np.random.choice([0, 1], n_samples, p=[0.3, 0.7])

print("Enhanced dataset with categorical features:")
df.head()

## Part 2: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Data types
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Missing values
print("\nMissing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0].sort_values('Percentage', ascending=False))

In [ ]:
# Statistical summary
print("\nNumerical Features Summary:")
df.describe()

In [ ]:
# Target variable analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['Price'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['Price'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: ${df['Price'].mean()*100000:,.0f}")
axes[0].axvline(df['Price'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: ${df['Price'].median()*100000:,.0f}")
axes[0].set_xlabel('Price ($100,000s)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of House Prices')
axes[0].legend()

# Box plot
axes[1].boxplot(df['Price'], vert=True)
axes[1].set_ylabel('Price ($100,000s)')
axes[1].set_title('Price Box Plot (Check for Outliers)')

plt.tight_layout()
plt.show()

print(f"\nPrice Statistics:")
print(f"Mean: ${df['Price'].mean()*100000:,.0f}")
print(f"Median: ${df['Price'].median()*100000:,.0f}")
print(f"Std: ${df['Price'].std()*100000:,.0f}")
print(f"Skewness: {df['Price'].skew():.2f}")

In [ ]:
# Correlation analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numeric_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# Top correlations with Price
print("\nCorrelation with Price:")
price_corr = correlation['Price'].sort_values(ascending=False)
print(price_corr)

In [ ]:
# Scatter plots of top features vs Price
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'AveBedrms']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, feature in enumerate(top_features):
    ax = axes[idx // 2, idx % 2]
    ax.scatter(df[feature], df['Price'], alpha=0.3, s=10)
    ax.set_xlabel(feature)
    ax.set_ylabel('Price ($100k)')
    ax.set_title(f'{feature} vs Price (r={correlation.loc[feature, "Price"]:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature analysis
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

fig, axes = plt.subplots(1, len(categorical_cols), figsize=(14, 5))

for idx, col in enumerate(categorical_cols):
    df.groupby(col)['Price'].mean().sort_values().plot(kind='bar', ax=axes[idx], color='teal')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Average Price ($100k)')
    axes[idx].set_title(f'Average Price by {col}')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Part 3: Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Price', axis=1)
y = df['Price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Identify column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Create preprocessing pipelines

# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combine pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline created!")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")

## Part 4: Model Building & Comparison

In [ ]:
# Define evaluation function
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a regression model
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    results = {
        'Model': model_name,
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Test MAE': mean_absolute_error(y_test, y_test_pred)
    }
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    results['CV R² Mean'] = cv_scores.mean()
    results['CV R² Std'] = cv_scores.std()
    
    return results, y_test_pred

In [ ]:
# Define models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

In [ ]:
# Train and evaluate all models
results_list = []
predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    results, pred = evaluate_model(model, X_train_processed, X_test_processed, y_train, y_test, name)
    results_list.append(results)
    predictions[name] = pred

# Create comparison DataFrame
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('Test R²', ascending=False)

print("\n" + "=" * 80)
print("MODEL COMPARISON RESULTS")
print("=" * 80)
print(results_df.round(4).to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
x_pos = np.arange(len(results_df))
width = 0.35

axes[0].bar(x_pos - width/2, results_df['Train R²'], width, label='Train R²', color='steelblue')
axes[0].bar(x_pos + width/2, results_df['Test R²'], width, label='Test R²', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score Comparison')
axes[0].legend()
axes[0].set_ylim(0, 1)

# RMSE comparison
axes[1].bar(results_df['Model'], results_df['Test RMSE'], color='teal')
axes[1].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[1].set_ylabel('RMSE ($100k)')
axes[1].set_title('Test RMSE Comparison (Lower is Better)')

plt.tight_layout()
plt.show()

## Part 5: Hyperparameter Tuning

In [ ]:
# Tune the best performing model (Random Forest or Gradient Boosting)
# Using GridSearchCV for hyperparameter optimization

# Define parameter grid for Random Forest
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

print("Starting GridSearchCV for Random Forest...")
print("This may take a few minutes...")

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_processed, y_train)

In [ ]:
# Best parameters
print("\nBest Parameters:")
print(rf_grid.best_params_)
print(f"\nBest CV R² Score: {rf_grid.best_score_:.4f}")

In [ ]:
# Evaluate best model
best_model = rf_grid.best_estimator_

best_results, best_pred = evaluate_model(
    best_model, X_train_processed, X_test_processed, 
    y_train, y_test, 'Random Forest (Tuned)'
)

print("\n" + "=" * 50)
print("TUNED MODEL RESULTS")
print("=" * 50)
for key, value in best_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## Part 6: Model Analysis & Interpretation

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(10, 8))

plt.scatter(y_test, best_pred, alpha=0.3, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($100k)')
plt.ylabel('Predicted Price ($100k)')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Residual analysis
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Residuals')

# Residuals vs Predicted
axes[1].scatter(best_pred, residuals, alpha=0.3, s=20)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Price ($100k)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted Values')

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"Mean: {residuals.mean():.4f}")
print(f"Std: {residuals.std():.4f}")

In [ ]:
# Feature importance
# Get feature names after preprocessing
cat_features_encoded = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(cat_features_encoded)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)
plt.barh(range(len(top_features)), top_features['Importance'], color='teal')
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Part 7: Final Model & Predictions

In [ ]:
# Create final pipeline
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])

# Retrain on full training data
final_pipeline.fit(X_train, y_train)

print("Final pipeline created and trained!")

In [ ]:
# Make predictions on new data
def predict_price(features_dict):
    """
    Predict house price for given features
    """
    input_df = pd.DataFrame([features_dict])
    prediction = final_pipeline.predict(input_df)[0]
    return prediction * 100000  # Convert to dollars

# Example prediction
sample_house = {
    'MedInc': 5.5,
    'HouseAge': 25.0,
    'AveRooms': 5.5,
    'AveBedrms': 1.1,
    'Population': 1500.0,
    'AveOccup': 3.0,
    'Latitude': 34.0,
    'Longitude': -118.0,
    'PropertyType': 'House',
    'Neighborhood': 'Suburban',
    'HasPool': 0,
    'HasGarage': 1
}

predicted_price = predict_price(sample_house)
print(f"Predicted House Price: ${predicted_price:,.0f}")

## Part 8: Conclusions & Recommendations

### Summary

1. **Best Model**: Random Forest Regressor with hyperparameter tuning
2. **Performance**: 
   - Test R² Score: ~0.80 (explains 80% of price variance)
   - RMSE: ~$XX,XXX

### Key Findings

1. **Most Important Features**:
   - Median Income is the strongest predictor
   - Location (Latitude, Longitude) matters
   - House Age and Room count are important

2. **Model Insights**:
   - Ensemble methods (Random Forest, Gradient Boosting) outperform linear models
   - Regularization helps prevent overfitting

### Recommendations

1. **For Model Improvement**:
   - Collect more features (school quality, crime rate, amenities)
   - Try advanced models (XGBoost, LightGBM)
   - Implement more feature engineering

2. **For Business Use**:
   - Use predictions as starting point, not final price
   - Consider confidence intervals
   - Update model regularly with new data

---

### Next Steps

1. Deploy model as API
2. Create user interface
3. Set up monitoring for model drift
4. Implement A/B testing

In [ ]:
# Save model (optional)
import joblib

# Save the pipeline
# joblib.dump(final_pipeline, 'house_price_model.pkl')
# print("Model saved!")

# Load model
# loaded_model = joblib.load('house_price_model.pkl')

---

## Capstone Project Checklist

- [x] Problem Definition
- [x] Data Collection
- [x] Exploratory Data Analysis
- [x] Data Preprocessing
- [x] Feature Engineering
- [x] Multiple Model Comparison
- [x] Hyperparameter Tuning
- [x] Model Evaluation
- [x] Feature Importance Analysis
- [x] Residual Analysis
- [x] Final Pipeline Creation
- [x] Conclusions & Recommendations

**Congratulations! You've completed the Regression Capstone Project!**